# 02 — Limpeza, ETL e Enriquecimento
**Olist Analytics Hub | Etapa 2 de 7**

**Objetivo:** Construir o *fact table* central (`orders_enriched`) que será usado em todos os notebooks subsequentes. Este é o pipeline de ETL do projeto.

---
### O que este notebook faz
1. Aplica limpeza e tipagem em cada tabela individualmente
2. Realiza feature engineering (datas, métricas de entrega, valor por pedido)
3. Mescla todas as tabelas num dataset enriquecido único
4. Salva o resultado em Parquet (comprimido, tipado, rápido)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.data_loader import load_all_raw, validate_datasets
from src.data_cleaner import build_orders_enriched, save_processed
from src.visualizations import setup_style, save_fig

setup_style()
pd.set_option('display.max_columns', None)
print('✓ Ambiente configurado.')

## 1. Carregamento dos Dados Brutos

In [ ]:
datasets = load_all_raw()
validate_datasets(datasets)

## 2. Construção do Fact Table Enriquecido

In [ ]:
print('Construindo orders_enriched...')
df = build_orders_enriched(datasets)
print(f'\n✓ orders_enriched: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
display(df.head(3))

In [ ]:
# Schema final
print('Colunas e tipos:')
print(df.dtypes.to_string())

## 3. Validação Pós-ETL

In [ ]:
# Taxa de correspondência entre tabelas
print('Pedidos no fact table:', df['order_id'].nunique())
print('Pedidos originais:', datasets['orders']['order_id'].nunique())

coverage = df['order_id'].nunique() / datasets['orders']['order_id'].nunique()
print(f'Cobertura: {coverage:.1%}')

In [ ]:
# Nulos nas colunas-chave
key_cols = [
    'order_id', 'customer_unique_id', 'order_status', 'order_purchase_timestamp',
    'revenue', 'freight', 'total_order_value', 'review_score',
    'delivery_days', 'customer_state', 'main_category'
]
null_summary = df[key_cols].isnull().sum().rename('nulos')
null_summary = pd.concat([null_summary, (null_summary / len(df) * 100).rename('%')], axis=1)
display(null_summary.sort_values('nulos', ascending=False))

In [ ]:
# Estatísticas das métricas de entrega
delivered = df[df['order_status'] == 'delivered']
print(f'Pedidos entregues: {len(delivered):,} ({len(delivered)/len(df):.1%} do total)')
print('\nEstatísticas de entrega (pedidos delivered):')
display(delivered[['delivery_days', 'days_to_estimated', 'delivery_delay_days']].describe().round(2))
print(f'\nEntregues no prazo: {delivered["delivered_on_time"].mean():.1%}')

## 4. Análise Rápida de Outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Dias de entrega
d_data = delivered['delivery_days'].dropna()
axes[0].hist(d_data, bins=80, color='#2E86AB', alpha=0.8, edgecolor='white')
axes[0].axvline(d_data.median(), color='#E94F37', linestyle='--', linewidth=2, label=f'Mediana: {d_data.median():.0f} dias')
axes[0].set_title('Distribuição de Dias para Entrega', fontweight='bold')
axes[0].set_xlabel('Dias')
axes[0].set_ylabel('Pedidos')
axes[0].legend()

# Atraso relativo à estimativa
delay_data = delivered['delivery_delay_days'].dropna()
colors_delay = ['#E94F37' if x > 0 else '#44BBA4' for x in delay_data.clip(-30, 60)]
axes[1].hist(delay_data.clip(-30, 60), bins=80, color='#A23B72', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='black', linestyle='-', linewidth=1.5)
axes[1].axvline(delay_data.median(), color='#F18F01', linestyle='--', linewidth=2, label=f'Mediana: {delay_data.median():.0f} dias')
axes[1].set_title('Atraso em relação à Estimativa\n(negativo = adiantado)', fontweight='bold')
axes[1].set_xlabel('Dias (- adiantado | + atrasado)')
axes[1].legend()

# Valor total do pedido
val_data = df['total_order_value'].dropna()
val_p99 = val_data.quantile(0.99)
axes[2].hist(val_data[val_data <= val_p99], bins=80, color='#F18F01', alpha=0.8, edgecolor='white')
axes[2].axvline(val_data.median(), color='#E94F37', linestyle='--', linewidth=2, label=f'Mediana: R$ {val_data.median():.2f}')
axes[2].set_title(f'Distribuição do Valor do Pedido (até p99)', fontweight='bold')
axes[2].set_xlabel('Valor (R$)')
axes[2].legend()

plt.tight_layout()
save_fig('02_distribuicoes_pos_etl')
plt.show()

## 5. Salvando o Dataset Processado

In [ ]:
print('Salvando orders_enriched.parquet...')
save_processed(df, 'orders_enriched')
print('\n✓ Pronto! Dataset central salvo em data/processed/')

## 6. Dicionário de Dados — orders_enriched

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `order_id` | str | Identificador único do pedido |
| `customer_id` | str | ID do cliente neste pedido |
| `customer_unique_id` | str | ID único do cliente real |
| `order_status` | str | Status final do pedido |
| `order_purchase_timestamp` | datetime | Data/hora da compra |
| `order_delivered_customer_date` | datetime | Data de entrega real |
| `order_estimated_delivery_date` | datetime | Data estimada de entrega |
| `n_items` | int | Número de itens no pedido |
| `revenue` | float | Soma dos preços (sem frete) |
| `freight` | float | Total de frete |
| `total_order_value` | float | revenue + freight |
| `main_category` | str | Categoria do produto principal (EN) |
| `payment_type` | str | Método de pagamento predominante |
| `payment_installments` | int | Número de parcelas |
| `payment_value` | float | Valor total pago |
| `review_score` | float | Nota de avaliação (1–5) |
| `has_comment` | bool | Cliente deixou comentário? |
| `customer_state` | str | Estado do cliente |
| `delivery_days` | float | Dias para entrega (compra → entrega) |
| `days_to_estimated` | float | Prazo estimado em dias |
| `delivery_delay_days` | float | Atraso relativo (+ = atrasado) |
| `delivered_on_time` | bool | Entregue no prazo? |
| `purchase_year` | int | Ano da compra |
| `purchase_month` | int | Mês da compra |
| `purchase_day_of_week` | str | Dia da semana |
| `purchase_hour` | int | Hora do dia |

---
**Próximo:** `03_analise_receita.ipynb`